# 01 — Baseline VLM : évaluation sur données RSNA réelles

Ce notebook évalue `baseline_predict` sur le **split dev RSNA** (150 cas étiquetés 3 classes).

| Paramètre | Valeur |
|-----------|--------|
| Dataset | RSNA Pneumonia Detection 2018 |
| Split | `dev` — 150 cas (50 normal / 50 suspected_opacity / 50 uncertain) |
| Modèle | `baseline_v1` (MedGemma-4B si `USE_MEDGEMMA=1`, sinon fallback règles) |
| Métriques | Accuracy, Macro-F1, Sensibilité, Spécificité, Latence, Taux d'incertitude |

> **Position non clinique.** Ce notebook est un outil pédagogique. Aucune sortie ne doit être interprétée comme un diagnostic.

In [ ]:
import sys, os, json, csv, time
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))

from src.inference import baseline_predict
from src.guardrails import apply_safety_guardrails
from src.metrics import summarize_metrics, macro_f1

CASES_CSV  = ROOT / 'data' / 'rsna' / 'cases.csv'
OUT_DIR    = ROOT / 'eval' / 'outputs'
OUT_DIR.mkdir(parents=True, exist_ok=True)

USE_MEDGEMMA = os.getenv('USE_MEDGEMMA', '0') == '1'
print(f'MedGemma actif : {USE_MEDGEMMA}')
print(f'Cases CSV      : {CASES_CSV}')
print(f'Output dir     : {OUT_DIR}')

## 1. Chargement du split dev RSNA

In [ ]:
with open(CASES_CSV, newline='', encoding='utf-8') as f:
    all_cases = list(csv.DictReader(f))

dev_cases = [c for c in all_cases if c['split'] == 'dev']
print(f'Total cases : {len(all_cases)}')
print(f'Dev split   : {len(dev_cases)} cas')

df_cases = pd.DataFrame(dev_cases)
print('\nDistribution des labels (ground truth) :')
print(df_cases['label'].value_counts().to_string())

## 2. Inférence baseline sur les 150 cas dev

In [ ]:
results = []
errors_run = []
start_total = time.perf_counter()

for i, case in enumerate(dev_cases):
    img_path = ROOT / case['image_path']
    try:
        pred = apply_safety_guardrails(baseline_predict(img_path))
        row = {
            'case_id'        : case['case_id'],
            'label'          : case['label'],
            'predicted_class': pred['predicted_class'],
            'confidence'     : pred['confidence'],
            'image_quality'  : pred['image_quality'],
            'latency_ms'     : pred['latency_ms'],
            'model_name'     : pred['model_name'],
            'correct'        : case['label'] == pred['predicted_class'],
            'justification'  : pred.get('justification', ''),
        }
        results.append(row)
    except Exception as exc:
        errors_run.append({'case_id': case['case_id'], 'error': str(exc)})
    
    if (i + 1) % 25 == 0:
        elapsed = time.perf_counter() - start_total
        print(f'  [{i+1:>3}/150]  {elapsed:.1f}s écoulées')

total_time = time.perf_counter() - start_total
print(f'\n✅ {len(results)} inférences en {total_time:.1f}s  |  {len(errors_run)} erreurs')
if errors_run:
    print('Erreurs :', errors_run[:3])

## 3. Métriques

In [ ]:
df = pd.DataFrame(results)

y_true = df['label'].tolist()
y_pred = df['predicted_class'].tolist()

accuracy = sum(t == p for t, p in zip(y_true, y_pred)) / len(y_true)
f1 = macro_f1(y_true, y_pred)

# Sensibilité : vrais positifs opacity parmi les vrais opacity
tp_opac = sum(t == 'suspected_opacity' and p == 'suspected_opacity' for t, p in zip(y_true, y_pred))
fn_opac = sum(t == 'suspected_opacity' and p != 'suspected_opacity' for t, p in zip(y_true, y_pred))
sensitivity = tp_opac / (tp_opac + fn_opac) if (tp_opac + fn_opac) else 0

# Spécificité : vrais négatifs normal parmi les vrais normaux
tn_norm = sum(t == 'normal' and p == 'normal' for t, p in zip(y_true, y_pred))
fp_norm = sum(t == 'normal' and p != 'normal' for t, p in zip(y_true, y_pred))
specificity = tn_norm / (tn_norm + fp_norm) if (tn_norm + fp_norm) else 0

uncertain_rate = sum(p == 'uncertain' for p in y_pred) / len(y_pred)
median_latency = float(np.median(df['latency_ms']))

metrics = {
    'model'          : df['model_name'].iloc[0],
    'n_cases'        : len(df),
    'accuracy'       : round(accuracy, 4),
    'macro_f1'       : round(f1, 4),
    'sensitivity_opacity': round(sensitivity, 4),
    'specificity_normal' : round(specificity, 4),
    'uncertain_rate' : round(uncertain_rate, 4),
    'median_latency_ms': median_latency,
    'total_time_s'   : round(total_time, 1),
}

print('=== MÉTRIQUES BASELINE — DEV RSNA (150 cas) ===')
for k, v in metrics.items():
    print(f'  {k:<28} {v}')

## 4. Visualisations

In [ ]:
from collections import Counter

CLASSES = ['normal', 'suspected_opacity', 'uncertain']

# Matrice de confusion
cm = np.zeros((3, 3), dtype=int)
for t, p in zip(y_true, y_pred):
    if t in CLASSES and p in CLASSES:
        cm[CLASSES.index(t)][CLASSES.index(p)] += 1

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# 1. Matrice de confusion
im = axes[0].imshow(cm, cmap='Blues')
axes[0].set_xticks(range(3)); axes[0].set_yticks(range(3))
axes[0].set_xticklabels(['normal', 'opacity', 'uncertain'], rotation=30, ha='right')
axes[0].set_yticklabels(['normal', 'opacity', 'uncertain'])
axes[0].set_xlabel('Prédit'); axes[0].set_ylabel('Réel')
axes[0].set_title(f'Matrice de confusion\nAccuracy={accuracy:.2%}  F1={f1:.2%}')
for i in range(3):
    for j in range(3):
        axes[0].text(j, i, cm[i, j], ha='center', va='center',
                     color='white' if cm[i, j] > cm.max()/2 else 'black', fontsize=12, fontweight='bold')

# 2. Distribution prédictions vs ground truth
gt_counts  = [sum(t == c for t in y_true) for c in CLASSES]
pred_counts = [sum(p == c for p in y_pred) for c in CLASSES]
x = np.arange(3); w = 0.35
axes[1].bar(x - w/2, gt_counts,  w, label='Ground truth', color='steelblue', alpha=0.8)
axes[1].bar(x + w/2, pred_counts, w, label='Prédit',       color='coral',     alpha=0.8)
axes[1].set_xticks(x); axes[1].set_xticklabels(['normal', 'opacity', 'uncertain'])
axes[1].set_ylabel('Count'); axes[1].set_title('Distribution GT vs Prédictions')
axes[1].legend(); axes[1].grid(axis='y', alpha=0.3)

# 3. Latence
axes[2].hist(df['latency_ms'], bins=20, color='seagreen', edgecolor='white', alpha=0.8)
axes[2].axvline(median_latency, color='red', linestyle='--', label=f'Médiane={median_latency:.0f}ms')
axes[2].set_xlabel('Latence (ms)'); axes[2].set_ylabel('Count')
axes[2].set_title('Distribution de la latence'); axes[2].legend()
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(str(OUT_DIR / 'baseline_rsna_dev_overview.png'), dpi=120, bbox_inches='tight')
plt.show()
print('Figure sauvegardée.')

## 5. Analyse d'erreurs (20 premiers cas mal classés)

In [ ]:
df_errors = df[~df['correct']].copy()
df_errors['error_type'] = df_errors.apply(
    lambda r: 'FP' if r['label'] == 'normal' and r['predicted_class'] == 'suspected_opacity'
    else ('FN' if r['label'] == 'suspected_opacity' and r['predicted_class'] == 'normal'
    else ('UA' if r['predicted_class'] == 'uncertain' else 'OTHER')), axis=1
)

print(f'Cas mal classés : {len(df_errors)}/{len(df)} ({len(df_errors)/len(df):.1%})')
print('\nRépartition par type d\'erreur :')
print(df_errors['error_type'].value_counts().to_string())

print('\n--- 10 premiers cas erronés ---')
print(df_errors[['case_id','label','predicted_class','confidence','image_quality','error_type']].head(10).to_string(index=False))

# Sauvegarde registre d'erreurs réel
error_rows = []
for _, row in df_errors.iterrows():
    error_rows.append({
        'case_id'     : row['case_id'],
        'ground_truth': row['label'],
        'prediction'  : row['predicted_class'],
        'error_type'  : row['error_type'],
        'confidence'  : row['confidence'],
        'image_quality': row['image_quality'],
        'comment'     : row['justification'][:120],
    })

error_path = OUT_DIR / 'error_register_rsna_baseline.csv'
with open(error_path, 'w', newline='', encoding='utf-8') as f:
    w = csv.DictWriter(f, fieldnames=error_rows[0].keys())
    w.writeheader(); w.writerows(error_rows)
print(f'\n✅ Registre d\'erreurs sauvegardé : {error_path}')

## 6. Sauvegarde des résultats

In [ ]:
# CSV complet
df.to_csv(OUT_DIR / 'rsna_dev_baseline_predictions.csv', index=False, encoding='utf-8')

# JSON métriques
with open(OUT_DIR / 'rsna_dev_baseline_metrics.json', 'w', encoding='utf-8') as f:
    json.dump(metrics, f, indent=2, ensure_ascii=False)

# JSON résultats complets (pour notebook 02)
df.to_json(OUT_DIR / 'rsna_dev_baseline_results.json', orient='records', indent=2, force_ascii=False)

print('Fichiers sauvegardés dans eval/outputs/ :')
for p in sorted(OUT_DIR.glob('rsna_dev_baseline*')):
    print(f'  {p.name}')

print('\n=== RÉSUMÉ FINAL ===')
print(f'  Accuracy          : {accuracy:.2%}')
print(f'  Macro-F1          : {f1:.2%}')
print(f'  Sensibilité opacité: {sensitivity:.2%}')
print(f'  Spécificité normal : {specificity:.2%}')
print(f'  Taux incertitude  : {uncertain_rate:.2%}')
print(f'  Latence médiane   : {median_latency:.0f} ms')